In [1]:
#pip install pm4py
import os
os.getcwd()

'C:\\Users\\obami\\Documents\\Python_Pra\\Thesis_PPM_2024-25'

In [2]:
#import and preprocess data
import numpy as np
import pandas as pd
import pm4py
import joblib

#Encode Prefix
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import LabelEncoder
from keras.preprocessing.sequence import pad_sequences

In [3]:
#df = pd.read_csv("dmd_df.csv")
#df = pd.read_csv("ptc_df.csv")
df = pd.read_csv('helpdesk_df.csv')
df.head()

,timestamp,activity,case_id
0,2010-01-13 08:40:25+00:00,assign seriousness,Case3608
1,2010-01-13 12:26:04+00:00,assign seriousness,Case2748
2,2010-01-13 12:30:37+00:00,assign seriousness,Case4284
3,2010-01-13 13:09:31+00:00,assign seriousness,Case1534
4,2010-01-13 17:25:25+00:00,assign seriousness,Case406


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21221 entries, 0 to 21220
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   timestamp  21221 non-null  object
 1   activity   21221 non-null  object
 2   case_id    21221 non-null  object
dtypes: object(3)
memory usage: 497.5+ KB


In [5]:
df["timestamp"] = pd.to_datetime(df["timestamp"]) # conversion from object to date type

In [6]:
#split data test
from split_train_test import split_train_test_temporal
train, test, fp_dict = split_train_test_temporal(df,0.2,"case_id","timestamp","preferred")

#split data validation
train, val, fp_dict = split_train_test_temporal(train,0.1,"case_id","timestamp","preferred")

In [7]:
from bos_eos import add_bos_eos_target
############## train data transformation ########################
train_prefix = add_bos_eos_target(train,"prefix")
train_target = add_bos_eos_target(train)

############## validation data transformation ###################
val_prefix = add_bos_eos_target(val,"prefix")
val_target = add_bos_eos_target(val)

############## test data transformation ########################
test_prefix = add_bos_eos_target(test,"prefix")
test_target = add_bos_eos_target(test)

In [8]:
train_prefix_copy = train_prefix.copy()
val_prefix_copy = val_prefix.copy()
test_prefix_copy = test_prefix.copy()

#get complete cases
cases = df["activity"].unique()

In [9]:
from prefix_trace import prefix_trace
############## train data transformation ########################
train_prefix_trace = prefix_trace(train_prefix)

############## validation data transformation ###################
val_prefix_trace = prefix_trace(val_prefix)

############## test data transformation ########################
test_prefix_trace = prefix_trace(test_prefix)

In [10]:
train_prefix_trace.head()

,case_id,prefix
0,Case10,[BOS]
1,Case10,"[BOS, assign seriousness]"
2,Case10,"[BOS, assign seriousness, take in charge ticket]"
3,Case10,"[BOS, assign seriousness, take in charge ticke..."
4,Case10,"[BOS, assign seriousness, take in charge ticke..."


In [11]:
from encoding import find_max_len, si_encoding

#all possible cases. I assume in a business all activities are already known and defined
cases = np.append(cases,"BOS")
cases = np.append(cases,"zos")

#find the maximum length of the longest case for padding
max_len = find_max_len(train_prefix_trace["prefix"],val_prefix_trace["prefix"],test_prefix_trace["prefix"])

from encoding import si_encoding
############## train data transformation ########################
train_prefix_trace_encoded, label_encoder = si_encoding(train_prefix_trace,cases,max_len)
train_target_encoded, a = si_encoding(train_target,cases,option = "target")

In [12]:
############## validation data transformation ###################
val_prefix_trace_encoded, a = si_encoding(val_prefix_trace,cases,max_len)
val_target_encoded, a = si_encoding(val_target,cases,option="target")

In [13]:
############## test data transformation ########################
test_prefix_trace_encoded, a = si_encoding(test_prefix_trace,cases,max_len)
test_target_encoded, a = si_encoding(test_target,cases,option="target")

In [14]:
train_target_encoded

array([[0., 1., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 1., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 1.]])

In [15]:
from dfg_probabilities import dfg_df

############ train #################
#get probability
probability = dfg_df(train_prefix,cases)

#encode labels for probability index
probability.index = label_encoder.transform(probability.index)
probability.columns = label_encoder.transform(probability.columns)

#reset index
probability.reset_index(inplace=True)
probability.rename(columns = {"index":"activity"},inplace=True)

#encode drop extra columns and encode activity
train_prefix["activity"] = label_encoder.transform(train_prefix["activity"])

#merge to get new dataframe
train_dfg_probability = pd.merge(train_prefix,probability,how="left",on="activity")
train_dfg_probability = train_dfg_probability.drop(columns = ["timestamp","case_id"])

In [16]:
############ validation ################################## 
#encode drop extra columns and encode activity
val_prefix["activity"] = label_encoder.transform(val_prefix["activity"])

#merge to get new dataframe. probability is same as train
val_dfg_probability = pd.merge(val_prefix,probability,how="left",on="activity")
val_dfg_probability = val_dfg_probability.drop(columns = ["timestamp","case_id"])

In [17]:
############ test ################################## 
#probability is combination of train and validation
train_val_prefix = pd.concat([train_prefix_copy,val_prefix])
probability = dfg_df(train_val_prefix,cases)

#encode drop extra columns and encode activity
test_prefix["activity"] = label_encoder.transform(test_prefix["activity"])

#encode labels for probability index
probability.index = label_encoder.transform(probability.index)
probability.columns = label_encoder.transform(probability.columns)

#reset index
probability.reset_index(inplace=True)
probability.rename(columns = {"index":"activity"},inplace=True)

#merge to get new dataframe
test_dfg_probability = pd.merge(test_prefix,probability,how="left",on="activity")
test_dfg_probability = test_dfg_probability.drop(columns = ["timestamp","case_id"])

In [ ]:
########################################### Help Desk ###########################################################################

##prefix data
np.save("helpdesk_train_prefix.npy",train_prefix_trace_encoded)
np.save("helpdesk_val_prefix.npy",val_prefix_trace_encoded)
np.save("helpdesk_test_prefix.npy",test_prefix_trace_encoded)

##probability data
train_dfg_probability.to_csv("helpdesk_train_dfg_probability.csv",index=False)
val_dfg_probability.to_csv("helpdesk_val_dfg_probability.csv",index=False)
test_dfg_probability.to_csv("helpdesk_test_dfg_probability.csv",index=False)

#target
np.save("helpdesk_train_target.npy",train_target_encoded)
np.save("helpdesk_val_target.npy",val_target_encoded)
np.save("helpdesk_test_target.npy",test_target_encoded)

#original data
train_target.to_csv("helpdesk_train_target_org.csv",index=False)
test_target.to_csv("helpdesk_test_target_org.csv",index=False)

In [ ]:
######################################### BPI Data ###############################################################################

##prefix data
np.save("ptc_train_prefix.npy",train_prefix_trace_encoded)
np.save("ptc_val_prefix.npy",val_prefix_trace_encoded)
np.save("ptc_test_prefix.npy",test_prefix_trace_encoded)

##probability data
train_dfg_probability.to_csv("ptc_train_dfg_probability.csv",index=False)
val_dfg_probability.to_csv("ptc_val_dfg_probability.csv",index=False)
test_dfg_probability.to_csv("ptc_test_dfg_probability.csv",index=False)

#target
np.save("ptc_train_target.npy",train_target_encoded)
np.save("ptc_val_target.npy",val_target_encoded)
np.save("ptc_test_target.npy",test_target_encoded)

#original data
train_target.to_csv("ptc_train_target_org.csv",index=False)
test_target.to_csv("ptc_test_target_org.csv",index=False)

In [ ]:
######################################### RMP Data ###############################################################################

##prefix data
np.save("dmd_train_prefix.npy",train_prefix_trace_encoded)
np.save("dmd_val_prefix.npy",val_prefix_trace_encoded)
np.save("dmd_test_prefix.npy",test_prefix_trace_encoded)

##probability data
train_dfg_probability.to_csv("dmd_train_dfg_probability.csv",index=False)
val_dfg_probability.to_csv("dmd_val_dfg_probability.csv",index=False)
test_dfg_probability.to_csv("dmd_test_dfg_probability.csv",index=False)

#target
np.save("dmd_train_target.npy",train_target_encoded)
np.save("dmd_val_target.npy",val_target_encoded)
np.save("dmd_test_target.npy",test_target_encoded)

#original data
train_target.to_csv("dmd_train_target_org.csv",index=False)
test_target.to_csv("dmd_test_target_org.csv",index=False)

In [ ]:
# ptc 
#joblib.dump(label_encoder, 'ptc_label_encoder.joblib'

In [ ]:
# Helpdesk 
joblib.dump(label_encoder, 'helpdesk_label_encoder.joblib')

In [19]:
# dmd 
joblib.dump(label_encoder, 'dmd_label_encoder.joblib')

['dmd_label_encoder.joblib']

In [21]:

# Helpdesk 
joblib.dump(label_encoder, 'helpdesk_label_encoder.joblib')

['helpdesk_label_encoder.joblib']

In [22]:
train_prefix_copy.head()

,timestamp,activity,case_id
0,2010-02-10 08:50:19+00:00,BOS,Case10
1,2010-02-10 08:50:20+00:00,assign seriousness,Case10
2,2010-03-19 08:47:06+00:00,take in charge ticket,Case10
3,2010-03-19 08:47:13+00:00,resolve ticket,Case10
4,2010-04-03 07:47:38+00:00,closed,Case10


In [23]:
cases

array(['assign seriousness', 'take in charge ticket', 'resolve ticket',
       'wait', 'create sw anomaly', 'closed', 'insert ticket',
       'schedule intervention', 'resolved', 'invalid', 'verified',
       'resolve sw anomaly', 'require upgrade', 'duplicate', 'BOS', 'zos'],
      dtype=object)

In [24]:
from dfg_probabilities import dfg_df

############ train #################
#get probability

probability = dfg_df(train_prefix_copy,cases)

In [25]:
probability.head()

,take in charge ticket,assign seriousness,resolve ticket,wait,create sw anomaly,schedule intervention,resolve sw anomaly,closed,resolved,verified,invalid,require upgrade,duplicate,insert ticket,zos
assign seriousness,0.824025,0.116381,0.047426,0.012168,0.000000,0.00000,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0
take in charge ticket,0.021829,0.000295,0.770206,0.195575,0.010619,0.00118,0.000295,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0
resolve ticket,0.042553,0.000000,0.029658,0.000645,0.000000,0.00000,0.000000,0.926821,0.000322,0.0,0.0,0.0,0.0,0.0,0.0
wait,0.661804,0.000000,0.273210,0.064987,0.000000,0.00000,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0
insert ticket,0.019048,0.980952,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0


In [26]:
#predict prefix, evaluate on target

In [27]:
next_activity = probability.idxmax(axis=1)
train_prefix_copy["next_activity"] = train_prefix_copy["activity"].map(next_activity)

In [28]:
train_prefix_copy["next_activity"].unique()

array(['assign seriousness', 'take in charge ticket', 'resolve ticket',
       'closed', 'invalid'], dtype=object)

In [29]:
#result

In [30]:
train_prefix_copy.head()

,timestamp,activity,case_id,next_activity
0,2010-02-10 08:50:19+00:00,BOS,Case10,assign seriousness
1,2010-02-10 08:50:20+00:00,assign seriousness,Case10,take in charge ticket
2,2010-03-19 08:47:06+00:00,take in charge ticket,Case10,resolve ticket
3,2010-03-19 08:47:13+00:00,resolve ticket,Case10,closed
4,2010-04-03 07:47:38+00:00,closed,Case10,closed


In [31]:
train_prefix_copy.fillna("none",inplace=True)

In [32]:
#result in report
from sklearn.metrics import multilabel_confusion_matrix, classification_report
report = classification_report(train_target["activity"],train_prefix_copy["next_activity"],zero_division=0)
print(report)

                       precision    recall  f1-score   support

   assign seriousness       0.95      0.88      0.91      3205
               closed       0.48      1.00      0.65      2877
    create sw anomaly       0.00      0.00      0.00        38
        insert ticket       0.00      0.00      0.00       105
              invalid       1.00      1.00      1.00         1
   resolve sw anomaly       0.00      0.00      0.00         5
       resolve ticket       0.77      0.85      0.81      3104
             resolved       0.00      0.00      0.00         1
schedule intervention       0.00      0.00      0.00         4
take in charge ticket       0.79      0.93      0.85      3390
             verified       0.00      0.00      0.00         1
                 wait       0.00      0.00      0.00       754
                  zos       0.00      0.00      0.00      2878

             accuracy                           0.70     16363
            macro avg       0.31      0.36      0.33 